# A/B Testing with Amazon Bedrock AgentCore

This notebook demonstrates how to run a **target-based A/B test** using Amazon Bedrock AgentCore. You will deploy two agent variants — a control and a treatment — then use the AgentCore Gateway to split live traffic between them and measure which performs better using automated online evaluation.

📚 [A/B Testing Overview](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/ab-testing.html) | [Target-Based Guide](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/ab-testing-target-based.html) | [Prerequisites](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/ab-testing-prereqs.html)

### Use Case: FixFirst Appliance Support Agent

**FixFirst** is a support agent that helps customers troubleshoot home appliances. We want to test whether a new model and prompt strategy improves helpfulness.

> **Note:** The agent implementation in this notebook has been slightly simplified to focus on the A/B testing mechanics rather than agent complexity.



| | Control (C) | Treatment (T1) |
|---|---|---|
| **Model** | Amazon Nova Lite | Claude Sonnet 4.5 |
| **Prompt style** | Conversational, one question at a time | Structured IDENTIFY/DIAGNOSE/RESOLVE methodology |
| **Evaluator** | Builtin.Helpfulness (LLM judge) | Builtin.Helpfulness (LLM judge) |

### What this notebook does

1. Checks prerequisites and installs missing dependencies
2. Packages and deploys both agent variants to AgentCore Runtime (zip deployment with OpenTelemetry)
3. Creates an AgentCore Gateway with HTTP targets for each variant
4. Configures online evaluation (Builtin.Helpfulness) for each variant
5. Creates an A/B test with 50/50 traffic split
6. Sends 20 appliance troubleshooting prompts through the gateway
7. Retrieves statistical results (mean scores, p-values, significance)
8. Cleans up all resources

**Estimated time:** ~30 minutes (including ~15 min wait for evaluation results)

**Prerequisites:** Python 3.12+, `uv`, Node.js, AWS CLI >= 2.34, CDK bootstrapped, Bedrock model access, `bash_kernel` (`pip install bash_kernel && python -m bash_kernel.install`).

**Note:** This notebook uses the **Bash kernel**. Select *Kernel → Change Kernel → Bash* if not already selected. 

See [README.md](README.md) for the full architecture diagram, project structure, and documentation links.

## How to Perform A/B Testing

A/B testing in AgentCore splits live production traffic between two variants and continuously evaluates performance with statistical significance. The **AgentCore Gateway** handles traffic routing — your agent code does not change.

### How it works

1. **Create an A/B test** — specify a Gateway, two variants (control & treatment), traffic weights, an online evaluation config, and an IAM execution role.
2. **Start the test** — the Gateway splits incoming traffic based on the runtime session ID. Assignment is sticky (same session → same variant).
3. **Online evaluation scores each session** — evaluators run against each completed session and map scores to variants.
4. **Statistical significance is computed** — per-evaluator metrics: mean score, p-value, confidence interval, significance flag. p < 0.05 = significant.
5. **Stop & deploy the winner** — route 100% traffic to the winning variant.

### Two Variant Patterns

AgentCore supports two A/B test patterns, each designed for different types of changes:

#### Pattern 1: Target-Based Variants

Use when the change involves **code changes, framework upgrades, or entirely different agent implementations**.

Each variant routes to a **different AgentCore Gateway target** pointing to a different runtime endpoint. This means you deploy two completely separate agents — they can use different models, different frameworks (e.g., Strands vs LangGraph), different tools, or different code logic.

- Variant configuration: `variantConfiguration.target` with the Gateway target name
- Evaluation: `perVariantOnlineEvaluationConfig` (one online eval config per variant — each endpoint has its own log group)
- Gateway filter: `gatewayFilter.targetPaths` scopes which paths the A/B test intercepts
- Each variant is fully independent — different code, model, dependencies

**Example use cases:**
- Testing a Strands agent vs a LangGraph agent
- Comparing Amazon Nova Lite (cheap/fast) vs Claude Sonnet (capable/expensive)
- Validating a major code refactor before full rollout
- Testing an agent with new tools vs one without

**Trade-offs:** Requires deploying and maintaining two separate runtimes. Higher infrastructure cost during testing, but gives maximum flexibility.

#### Pattern 2: Configuration Bundle Variants

Use when the change is **purely configuration** (system prompt, model ID, tool descriptions, temperature).

Both variants run on the **same AgentCore Runtime** with different configuration bundle versions. The Gateway injects the bundle reference via W3C baggage headers. The agent code reads the active bundle at runtime using a `BeforeModelCallEvent` hook and applies the configuration dynamically.

- Variant configuration: `variantConfiguration.configurationBundle` with the bundle version
- Evaluation: `sharedOnlineEvaluationConfig` (single shared config — both variants share the same log group)
- No separate endpoints needed — one runtime handles both variants
- Lighter weight — no extra infrastructure, just different config values

**Example use cases:**
- Testing a new system prompt (e.g., conversational vs structured)
- Comparing model parameters (temperature, top-p)
- A/B testing tool descriptions to see which phrasing triggers better tool use
- Iterating on prompt engineering without redeploying code

**Trade-offs:** Limited to configuration-only changes. The agent code must be designed to read config bundles (requires the `BeforeModelCallEvent` hook). But much cheaper and faster to iterate.

#### Choosing a Pattern

| Aspect | Target-based | Configuration bundle |
|--------|-------------|---------------------|
| What varies | Entire runtime (code, framework, model) | System prompt, model parameters |
| Routing | Different targets per variant | Same target, different config bundles |
| Evaluation config | One per variant | Single shared config |
| Infrastructure cost | 2 runtimes | 1 runtime |
| Iteration speed | Slow (redeploy) | Fast (update bundle) |
| When to use | Code/model changes | Prompt engineering |

This notebook demonstrates **Pattern 1: Target-Based Variants**.

## 1. Check Prerequisites

Set `HOME_DIR` to the root of the cloned repository (default: `~/sample-aws-agentops-fix-first-agent`). Then verify all required tools, credentials, and services are available.

In [ ]:
export HOME_DIR=~/sample-aws-agentops-fix-first-agent
cd $HOME_DIR/ab_testing
$HOME_DIR/ab_testing/scripts/check_prerequisites.sh

## 2. Deploy Both Agents

We deploy both agent variants to AgentCore Runtime using **direct code deployment** (zip). Each agent is packaged with:
- Python dependencies compiled for Linux arm64 (AgentCore Runtime architecture)
- A `bin/opentelemetry-instrument` script that enables the OTel collector sidecar
- The `entryPoint: ['opentelemetry-instrument', 'main.py']` configuration in CDK

The CDK stack also sets `requestHeaderConfiguration` to allow `baggage` and `traceparent` headers — required for the A/B test to propagate experiment metadata via W3C baggage headers.

📚 [AgentCore Runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability-get-started.html) | [OpenTelemetry Setup](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability-telemetry.html)

### Control vs Treatment: What We're Testing

Both agents solve the same problem (appliance troubleshooting) but differ in two dimensions:

**Control (`fixFirstAgent_Control_Agent`):**
- Uses **Amazon Nova Lite** — fast, cost-effective, good for simple conversational tasks
- Prompt style: *"Keep responses short and conversational. Ask one question at a time."*
- Hypothesis: friendly but may lack depth for complex diagnostics

**Treatment (`fixFirstAgent_Treatment_Agent`):**
- Uses **Claude Sonnet 4.5** — more capable reasoning model, higher cost
- Prompt style: *"Use structured methodology: IDENTIFY, DIAGNOSE, RESOLVE. Keep responses to 2-3 sentences max."*
- Hypothesis: structured approach should produce more helpful, actionable responses

### What We Expect to Find

The `Builtin.Helpfulness` evaluator (an LLM judge) scores each session on how helpful the agent's response was. We expect:
- The treatment (Claude + structured prompt) to score higher on helpfulness
- But we need statistical significance (p < 0.05) to be confident the difference isn't random noise
- If the improvement is significant, we can justify the higher cost of Claude for production deployment
- If not significant, we keep the cheaper Nova Lite model

### 2.1 Package Agents

In [ ]:
$HOME_DIR/ab_testing/target_based_variants/scripts/package_agents.sh $HOME_DIR/ab_testing/target_based_variants/agents

### 2.2 Deploy Runtimes + Evaluation Configs

Deploys the `fixFirstAgent-ABTestingStack` via CDK. This single stack creates:
- Two AgentCore Runtimes (control + treatment) with zip artifacts from S3
- Two Online Evaluation Configs (`Builtin.Helpfulness`) pointing to each runtime's log group
- IAM execution roles for runtimes, evaluator, and gateway
- SSM parameters storing all resource ARNs

📚 [Online Evaluation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/create-online-evaluations.html) | [Evaluation Results](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/results-and-output.html)

In [ ]:
cd $HOME_DIR/ab_testing/target_based_variants/cdk_ab_testing
echo "Installing CDK dependencies..."
npm install
echo "Starting CDK deploy (this may take 5-10 minutes with no output)..."
npx --yes cdk deploy fixFirstAgent-ABTestingStack --require-approval never

### 2.3 Wait for Runtimes to Become READY

AgentCore Runtimes take 1-3 minutes to provision after stack creation.

In [ ]:
REGION=${AWS_REGION:-$(aws configure get region 2>/dev/null)}
REGION=${REGION:-us-east-1}
CONTROL_ARN=$(aws ssm get-parameter --name /fixFirstAgent/control-runtime-arn --query Parameter.Value --output text --region $REGION)
REFINED_ARN=$(aws ssm get-parameter --name /fixFirstAgent/refined-runtime-arn --query Parameter.Value --output text --region $REGION)
CONTROL_ID=$(echo "$CONTROL_ARN" | awk -F/ '{print $NF}')
REFINED_ID=$(echo "$REFINED_ARN" | awk -F/ '{print $NF}')

for NAME_ID in "Control:$CONTROL_ID" "Treatment:$REFINED_ID"; do
    NAME=${NAME_ID%%:*}
    RID=${NAME_ID#*:}
    echo "Waiting for $NAME runtime ($RID)..."
    for i in $(seq 1 30); do
        STATUS=$(aws bedrock-agentcore-control get-agent-runtime --agent-runtime-id "$RID" --region $REGION --query status --output text 2>/dev/null)
        if [ "$STATUS" = "READY" ]; then
            echo "  $NAME is READY"
            break
        fi
        echo "  [$i/30] $STATUS"
        sleep 20
    done
    if [ "$STATUS" != "READY" ]; then
        echo "ERROR: $NAME not READY after 10 minutes"
        exit 1
    fi
done
echo "Both runtimes READY."

## 3. Target-Based A/B Testing

Now we set up the A/B test infrastructure:
- An AgentCore Gateway with IAM auth
- Two HTTP targets (control + treatment) pointing to the deployed runtimes
- A 50/50 A/B test with per-variant online evaluation

### How traffic flows through the system

<pre>
Client (SigV4-signed request)
    |
    v
+---------------------------------------------------------------+
|                  AgentCore Gateway (IAM Auth)                 |
|                                                               |
|   +---------------------------------------------------------+ |
|   |            A/B Test (50/50 traffic split)               | |
|   |     Assigns session -> variant (sticky routing)         | |
|   +---------------------------+-----------------------------+ |
+------------------------------|--------------------------------+
                               |
              +----------------+----------------+
              |                                 |
    +---------+---------+          +------------+----------+
    | Target: control   |          | Target: treatment     |
    +---------+---------+          +------------+----------+
              |                                 |
    +---------+---------+          +------------+----------+
    | AgentCore Runtime |          | AgentCore Runtime     |
    | Amazon Nova Lite  |          | Claude Sonnet 4.5     |
    | (Control - C)     |          | (Treatment - T1)      |
    +---------+---------+          +------------+----------+
              |                                 |
         OTel spans                        OTel spans
              |                                 |
    +---------+---------+          +------------+----------+
    | Online Eval (C)   |          | Online Eval (T1)      |
    | Builtin.          |          | Builtin.              |
    | Helpfulness       |          | Helpfulness           |
    +---------+---------+          +------------+----------+
              |                                 |
              +-----------------+---------------+
                                |
              +-----------------+---------------+
              |   A/B Test Aggregation          |
              |   mean, p-value, CI,            |
              |   significance, recommendation  |
              +---------------------------------+
</pre>

**Step by step:**

1. The **client** sends a SigV4-signed HTTP request to the gateway endpoint. Each request includes a unique session ID.
2. The **A/B test** inside the gateway inspects the session ID and assigns it to either the control (C) or treatment (T1) variant. Assignment is sticky — the same session ID always routes to the same variant.
3. The **gateway** forwards the request to the corresponding **target** (`control` or `treatment`), injecting W3C `baggage` headers containing the experiment ARN and variant name.
4. The **AgentCore Runtime** processes the request. The OTel collector sidecar captures spans and stamps them with the experiment metadata from the baggage headers.
5. After 15 minutes of inactivity, the session is considered complete. The **online evaluator** (`Builtin.Helpfulness`) scores the session using an LLM judge.
6. The **A/B test aggregation pipeline** collects scores from both variants and computes statistics: mean score per variant, p-value, confidence interval, and whether the difference is statistically significant.

📚 [Gateway Concepts](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-core-concepts.html) | [HTTP Runtime Targets](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-target-http-runtime.html)

### 3.1 Deploy Gateway, Targets, and A/B Test

In [ ]:
REGION=${AWS_REGION:-$(aws configure get region 2>/dev/null)}
REGION=${REGION:-us-east-1}
CONTROL_RUNTIME_ARN=$(aws ssm get-parameter --name /fixFirstAgent/control-runtime-arn --query Parameter.Value --output text --region $REGION)
REFINED_RUNTIME_ARN=$(aws ssm get-parameter --name /fixFirstAgent/refined-runtime-arn --query Parameter.Value --output text --region $REGION)
CONTROL_EVAL_ARN=$(aws ssm get-parameter --name /fixFirstAgent/control-eval-arn --query Parameter.Value --output text --region $REGION)
TREATMENT_EVAL_ARN=$(aws ssm get-parameter --name /fixFirstAgent/treatment-eval-arn --query Parameter.Value --output text --region $REGION)

cd $HOME_DIR/ab_testing/target_based_variants/cdk_ab_gateway
echo "Installing CDK dependencies..."
npm install
echo "Starting CDK deploy (this may take 5-10 minutes with no output)..."
npx --yes cdk deploy fixFirstAgent-ABGatewayStack --require-approval never \
    -c "controlRuntimeArn=$CONTROL_RUNTIME_ARN" \
    -c "refinedRuntimeArn=$REFINED_RUNTIME_ARN" \
    -c "controlEvalArn=$CONTROL_EVAL_ARN" \
    -c "treatmentEvalArn=$TREATMENT_EVAL_ARN"

### 3.2 Send Traffic Through Gateway

Sends 20 appliance troubleshooting prompts through the gateway using SigV4-signed HTTP requests. Each request gets a unique session ID. The A/B test assigns each session to control or treatment based on the 50/50 weight.

The gateway injects `baggage` headers into the forwarded request, which the runtime SDK's `BaggageSpanProcessor` reads to stamp all OTel spans with the experiment ARN and variant name.

In [ ]:
REGION=${AWS_REGION:-$(aws configure get region 2>/dev/null)}
REGION=${REGION:-us-east-1}
GATEWAY_URL=$(aws ssm get-parameter --name /fixFirstAgent/ab-gateway-url --query Parameter.Value --output text --region $REGION)
cd $HOME_DIR/ab_testing
$HOME_DIR/ab_testing/scripts/send_traffic.sh "$GATEWAY_URL" "$REGION" $HOME_DIR/ab_testing/prompts.txt

### 3.3 Check A/B Test Results

Retrieves the A/B test results. Re-run after ~20 minutes if results haven't appeared yet.

**Interpreting results:**
- `p-value < 0.05` + positive `percentChange` = treatment is significantly better
- `p-value < 0.05` + negative `percentChange` = treatment is significantly worse
- `p-value >= 0.05` = not enough evidence, continue collecting samples

📚 [Managing A/B Tests](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/ab-testing-manage.html)

In [ ]:
python3 $HOME_DIR/ab_testing/scripts/check_ab_results.py || python $HOME_DIR/ab_testing/scripts/check_ab_results.py

### 3.4 Stop the A/B Test

Stops the A/B test. The gateway reverts to default routing (no traffic splitting).

In [ ]:
REGION=${AWS_REGION:-$(aws configure get region 2>/dev/null)}
REGION=${REGION:-us-east-1}
AB_TEST_ID=$(aws ssm get-parameter --name /fixFirstAgent/ab-test-id --query Parameter.Value --output text --region $REGION)
aws bedrock-agentcore update-ab-test --ab-test-id "$AB_TEST_ID" --execution-status STOPPED --region $REGION
echo "A/B test $AB_TEST_ID stopped."

### 3.5 Cleanup

Removes all resources: A/B test, gateway, targets, IAM role, CDK stacks.

In [ ]:
REGION=${AWS_REGION:-$(aws configure get region 2>/dev/null)}
export APP_NAME=fixFirstAgent
export AWS_REGION=${REGION:-us-east-1}
$HOME_DIR/ab_testing/target_based_variants/scripts/cleanup_all.sh

## Running the Full Workflow Without a Notebook

You can run the entire A/B testing workflow end-to-end as a single script:

```bash
./run_target_ab_testing.sh
```

This performs all steps (prerequisites → package → deploy → wait → send traffic → poll for results) and prints the final recommendation.